In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
fname_lowT = "./raw_experiment/p7_martin.csv"
fname_highT = "./raw_experiment/martin_high_temp.csv"

df_lowT = pd.read_csv(fname_lowT)
df_highT = pd.read_csv(fname_highT)

In [3]:
df_lowT_filtered = df_lowT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_lowT_filtered.rename(columns={"ACTIVITY": "activity_37"}, inplace=True)
df_lowT_filtered["activity_37"] = df_lowT_filtered["activity_37"].replace([np.inf, -np.inf], np.nan)
df_lowT_filtered["activity_37"] = df_lowT_filtered["activity_37"].fillna(-5)

df_highT_filtered = df_highT.filter(["id", "seq", "DIST", "ACTIVITY"])
df_highT_filtered.rename(columns={"ACTIVITY": "activity_60"}, inplace=True)
df_highT_filtered["activity_60"] = df_highT_filtered["activity_60"].replace([np.inf, -np.inf], np.nan)
df_highT_filtered["activity_60"] = df_highT_filtered["activity_60"].fillna(-5)

In [4]:
df_filtered = df_lowT_filtered.merge(
    df_highT_filtered,
    on=["id"],                                 
)
df = df_filtered.filter(["id", "seq_x", "DIST_x", "activity_37", "activity_60"])
df.rename(columns={
    "seq_x": "seq",
    "DIST_x": "dist_azo",
}, inplace=True)

print(len(df))
# remove Giovanni sequences
def test_fn(s):
    return "reint" not in s

df = df[df["id"].map(test_fn)]
print(len(df))

4493
3500


In [5]:
df.head()

,id,seq,dist_azo,activity_37,activity_60
0,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,28,-2.982426,-3.578521
1,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_696,GUGCAUUCCGUCGGGAAACCACCGAAUGGAUGUUGUCAAAUUCGGU...,25,-2.646676,-3.208042
2,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_910,GUGCAUUCCGCCGGGAAACCACUGAAUGGCUGUUGUCAAAUUCGGU...,27,-0.592556,-2.084677
3,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_545,GUGCAUUCCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,29,-2.463253,-3.154773
4,MARTIN_wt_2_mut_lowDCA_lowSB_beta1.25_sb-3.00_...,GUGCAUUUCGUCGGGAAACCACCGAAUGGCUGUUGUCAAAUUCGGU...,25,-1.349940,-2.098292


In [6]:
wt_id = []
region = []

for index, row in df.iterrows():
    if row["dist_azo"] >= 100:
        region.append("control")
        wt_id.append(np.nan)
    else:
        id = row["id"]
        if ("lowDCA" in id) and ("lowSB" in id):
            region.append("lowDCA_lowSB")
        elif ("highDCA" in id) and ("highSB" in id):
            region.append("highDCA_highSB")
        elif ("lowDCA" in id) and ("highSB" in id):
            region.append("lowDCA_highSB")
        else:
            print(id)
        wt_id.append(id.split("_")[2])
        

df["wt_id"] = wt_id
df["region"] = region

In [7]:
df.to_csv("experiment_results.csv", index=False)